In [ ]:
states = ['NOUN','VERB','PRON','ADJ']

In [ ]:
obs = ['he','reads','books','she','eats','pizza','they','enjoy','football','are','educational','is','tasty']

In [ ]:
sp = {'NOUN':0.4,'VERB':0 , 'PRON':0.6,'ADJ':0}

In [ ]:
tp = {
    'NOUN': {'NOUN': 0, 'VERB': 0.6, 'PRON': 0, 'ADJ': 0.4},
    'VERB': {'NOUN': 1, 'VERB': 0, 'PRON': 1, 'ADJ': 0},
    'PRON': {'NOUN': 0, 'VERB': 0, 'PRON': 0, 'ADJ': 0},
    'ADJ': {'NOUN': 0, 'VERB': 0.4, 'PRON': 0, 'ADJ': 0}
}


In [ ]:

ep = {
    'NOUN': {'he': 0, 'reads': 0, 'books': 0, 'she': 0, 'eats': 0, 'pizza': 0, 'they': 0, 'enjoy': 0, 'football': 0.2, 'are': 0, 'educational': 0, 'is': 0, 'tasty': 0},
    'VERB': {'he': 0, 'reads': 0, 'books': 0, 'she': 0, 'eats': 0, 'pizza': 0, 'they': 0, 'enjoy': 0, 'football': 0, 'are': 0, 'educational': 0, 'is': 0.2, 'tasty': 0},
    'PRON': {'he': 0.33, 'reads': 0.2, 'books': 0, 'she': 0, 'eats': 0, 'pizza': 0, 'they': 0, 'enjoy': 0, 'football': 0, 'are': 0, 'educational': 0, 'is': 0, 'tasty': 0},
    'ADJ': {'he': 0, 'reads': 0, 'books': 0, 'she': 0, 'eats': 0.2, 'pizza': 0, 'they': 0, 'enjoy': 0, 'football': 0, 'are': 0, 'educational': 0.5, 'is': 0, 'tasty': 0.5}
}


In [ ]:
# prompt: import numpy as np
# states = ['NOUN', 'VERB', 'PRON', 'ADJ']
# obs = ['he', 'reads', 'books', 'she', 'eats', 'pizza', 'they', 'enjoy', 'football', 'are', 'educational', 'is', 'tasty']
# sp = {'NOUN': 0.4, 'VERB': 0, 'PRON': 0.6, 'ADJ': 0}
# tp = {
#     'NOUN': {'NOUN': 0, 'VERB': 0.6, 'PRON': 0, 'ADJ': 0.4},
#     'VERB': {'NOUN': 1, 'VERB': 0, 'PRON': 0, 'ADJ': 0},
#     'PRON': {'NOUN': 0, 'VERB': 0, 'P

import numpy as np

states = ['NOUN', 'VERB', 'PRON', 'ADJ']
obs = ['he', 'reads', 'books', 'she', 'eats', 'pizza', 'they', 'enjoy', 'football', 'are', 'educational', 'is', 'tasty']

sp = {'NOUN': 0.4, 'VERB': 0, 'PRON': 0.6, 'ADJ': 0}
tp = {
    'NOUN': {'NOUN': 0, 'VERB': 0.6, 'PRON': 0, 'ADJ': 0.4},
    'VERB': {'NOUN': 1, 'VERB': 0, 'PRON': 0, 'ADJ': 0},
    'PRON': {'NOUN': 0, 'VERB': 0, 'PRON': 0, 'ADJ': 0},
    'ADJ': {'NOUN': 0, 'VERB': 0.4, 'PRON': 0, 'ADJ': 0}
}

ep = {
    'NOUN': {'he': 0, 'reads': 0, 'books': 0, 'she': 0, 'eats': 0, 'pizza': 0, 'they': 0, 'enjoy': 0, 'football': 0.1, 'are': 0, 'educational': 0.2, 'is': 0, 'tasty': 0},
    'VERB': {'he': 0, 'reads': 0.2, 'books': 0, 'she': 0, 'eats': 0.3, 'pizza': 0, 'they': 0, 'enjoy': 0, 'football': 0.1, 'are': 0, 'educational': 0, 'is': 0.2, 'tasty': 0},
    'PRON': {'he': 0.33, 'reads': 0.2, 'books': 0, 'she': 0.33, 'eats': 0, 'pizza': 0, 'they': 0.33, 'enjoy': 0, 'football': 0, 'are': 0, 'educational': 0, 'is': 0, 'tasty': 0},
    'ADJ': {'he': 0, 'reads': 0, 'books': 0.2, 'she': 0, 'eats': 0.2, 'pizza': 0, 'they': 0, 'enjoy': 0, 'football': 0, 'are': 0, 'educational': 0.5, 'is': 0, 'tasty': 0.5}
}


def viterbi(obs_seq):
  """
  Implements the Viterbi algorithm for HMM.

  Args:
    obs_seq: A list of observations.

  Returns:
    A list of most likely hidden states.
  """

  n_states = len(states)
  n_obs = len(obs_seq)

  viterbi_table = np.zeros((n_states, n_obs))
  backpointer = np.zeros((n_states, n_obs), dtype=np.int32)

  # Initialization
  for s in range(n_states):
    viterbi_table[s, 0] = sp[states[s]] * ep[states[s]].get(obs_seq[0], 0)
    backpointer[s, 0] = 0

  # Recursion
  for t in range(1, n_obs):
    for s in range(n_states):
      max_prob = 0
      best_prev_state = 0
      for prev_s in range(n_states):
        prob = viterbi_table[prev_s, t - 1] * tp[states[prev_s]].get(states[s], 0) * ep[states[s]].get(obs_seq[t], 0)
        if prob > max_prob:
          max_prob = prob
          best_prev_state = prev_s

      viterbi_table[s, t] = max_prob
      backpointer[s, t] = best_prev_state

  # Termination
  best_path_prob = np.max(viterbi_table[:, -1])
  best_path_pointer = np.argmax(viterbi_table[:, -1])

  # Backtracking
  best_path = [best_path_pointer]
  for t in range(n_obs - 1, 0, -1):
    best_path_pointer = backpointer[best_path_pointer, t]
    best_path.insert(0, best_path_pointer)

  return [states[i] for i in best_path], viterbi_table

sentence = ["he", "eats", "football"]

best_path, viterbi_table = viterbi(sentence)

print("Viterbi Table:")
print(viterbi_table)

print("\nMost Likely Hidden State Sequence:")
print(best_path)


Viterbi Table:
[[0.    0.    0.   ]
 [0.    0.    0.   ]
 [0.198 0.    0.   ]
 [0.    0.    0.   ]]

Most Likely Hidden State Sequence:
['NOUN', 'NOUN', 'NOUN']


In [ ]:
# prompt: noun noun noun is wrong correct the values to get correct output for he eats football

import numpy as np

states = ['NOUN', 'VERB', 'PRON', 'ADJ']
obs = ['he', 'reads', 'books', 'she', 'eats', 'pizza', 'they', 'enjoy', 'football', 'are', 'educational', 'is', 'tasty']

sp = {'NOUN': 0.4, 'VERB': 0, 'PRON': 0.6, 'ADJ': 0}
tp = {
    'NOUN': {'NOUN': 0, 'VERB': 0.6, 'PRON': 0, 'ADJ': 0.4},
    'VERB': {'NOUN': 0.8, 'VERB': 0, 'PRON': 0, 'ADJ': 0.2},  # Adjusted transition probabilities
    'PRON': {'NOUN': 0, 'VERB': 0.7, 'PRON': 0, 'ADJ': 0.3},  # Adjusted transition probabilities
    'ADJ': {'NOUN': 0, 'VERB': 0.4, 'PRON': 0, 'ADJ': 0}
}

ep = {
    'NOUN': {'he': 0, 'reads': 0, 'books': 0.2, 'she': 0, 'eats': 0, 'pizza': 0.3, 'they': 0, 'enjoy': 0, 'football': 0.1, 'are': 0, 'educational': 0, 'is': 0, 'tasty': 0},
    'VERB': {'he': 0, 'reads': 0.2, 'books': 0, 'she': 0, 'eats': 0.3, 'pizza': 0, 'they': 0, 'enjoy': 0, 'football': 0.1, 'are': 0, 'educational': 0, 'is': 0.2, 'tasty': 0},
    'PRON': {'he': 0.33, 'reads': 0.2, 'books': 0, 'she': 0.33, 'eats': 0, 'pizza': 0, 'they': 0.33, 'enjoy': 0, 'football': 0, 'are': 0, 'educational': 0, 'is': 0, 'tasty': 0},
    'ADJ': {'he': 0, 'reads': 0, 'books': 0, 'she': 0, 'eats': 0, 'pizza': 0, 'they': 0, 'enjoy': 0, 'football': 0, 'are': 0, 'educational': 0, 'is': 0, 'tasty': 0}
}


def viterbi(obs_seq):
  """
  Implements the Viterbi algorithm for HMM.

  Args:
    obs_seq: A list of observations.

  Returns:
    A list of most likely hidden states.
  """

  n_states = len(states)
  n_obs = len(obs_seq)

  viterbi_table = np.zeros((n_states, n_obs))
  backpointer = np.zeros((n_states, n_obs), dtype=np.int32)

  # Initialization
  for s in range(n_states):
    viterbi_table[s, 0] = sp[states[s]] * ep[states[s]].get(obs_seq[0], 0)
    backpointer[s, 0] = 0

  # Recursion
  for t in range(1, n_obs):
    for s in range(n_states):
      max_prob = 0
      best_prev_state = 0
      for prev_s in range(n_states):
        prob = viterbi_table[prev_s, t - 1] * tp[states[prev_s]].get(states[s], 0) * ep[states[s]].get(obs_seq[t], 0)
        if prob > max_prob:
          max_prob = prob
          best_prev_state = prev_s

      viterbi_table[s, t] = max_prob
      backpointer[s, t] = best_prev_state

  # Termination
  best_path_prob = np.max(viterbi_table[:, -1])
  best_path_pointer = np.argmax(viterbi_table[:, -1])

  # Backtracking
  best_path = [best_path_pointer]
  for t in range(n_obs - 1, 0, -1):
    best_path_pointer = backpointer[best_path_pointer, t]
    best_path.insert(0, best_path_pointer)

  return [states[i] for i in best_path], viterbi_table

sentence = ["he", "eats", "football"]

best_path, viterbi_table = viterbi(sentence)

print("Viterbi Table:")
print(viterbi_table)

print("\nMost Likely Hidden State Sequence:")
print(best_path)


Viterbi Table:
[[0.        0.        0.0033264]
 [0.        0.04158   0.       ]
 [0.198     0.        0.       ]
 [0.        0.        0.       ]]

Most Likely Hidden State Sequence:
['PRON', 'VERB', 'NOUN']
